# DuckDB, an analytical database in a library

Day 1, before Lab 1.1 · CPU only. Finance asks for revenue per store per day. You ask the app database and pandas,
watch both struggle, then learn the engine every lab of this course runs on.

**The data:** a large copy of Qafila's orders table, built by the setup cell, and a few daily files small enough to
check by eye. Lab 1.1 points the same tools at the real sources.

🤔 asks you to predict before you run a cell · 🔍 says what the output means · ➕ sections are for your own pace.

**Contents**
- [ ] 1 · The problem: analytics on the app database
- [ ] 2 · The mental model: in-process, columnar, vectorized
- [ ] 3 · A query, a connection, a database file
- [ ] 4 · Files where they are: CSV, globs, JSON, Parquet
- [ ] 5 · Tables, views, other databases, and one writer at a time
- [ ] 6 · Python in and out: DataFrames, relations, parameters
- [ ] 7 · Friendly SQL ➕
- [ ] 8 · Read the plan: EXPLAIN and EXPLAIN ANALYZE ➕
- [ ] 9 · Summary

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/data-engineering.git"
    if not Path("/content/data-engineering").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/data-engineering"], check=True)
    os.chdir("/content/data-engineering/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.4", "polars>=1.30"], check=True)
else:
    while not Path("de_utils.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # tools/ and solutions/tools/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import re
import sqlite3

import duckdb
import pandas as pd
from IPython.display import display
import de_utils as U
import de_viz as V

pd.set_option("display.width", 160)


def first_error(stderr):
    """The error line of a failed subprocess, with long file paths shortened."""
    stderr = re.sub(r"\x1b\[[0-9;]*m", "", stderr)        # no terminal colours
    line = next((l for l in stderr.splitlines() if "Error" in l), stderr.strip()[-200:])
    return re.sub(r'"[^"]*[\\/]([^"\\/]+)"', r'"…/\1"', line)

# Rerunning THIS cell starts again from an empty folder (it only deletes this notebook's own outputs).
OUT = U.lab_dir("tools_duckdb", reset=True)
F = OUT.as_posix()                     # DuckDB likes forward slashes, also on Windows
APP = OUT / "app.sqlite"               # a stand-in for the app's database
N = U.pick(1_000_000, 200_000)         # orders in it (fewer in quick mode)

# The app database: one big orders table, as the app stores it (timestamps as text). How this cell works is §5.
with duckdb.connect() as build:
    build.sql(f"ATTACH '{APP.as_posix()}' AS app (TYPE sqlite)")
    build.sql(f"""
        CREATE TABLE app.orders AS
        SELECT i AS order_id,
               ['RUH-04', 'RUH-05', 'JED-03', 'DMM-01', 'MED-02'][1 + i % 5] AS store_id,
               strftime(TIMESTAMP '2026-08-01' + to_seconds(i * 5 * 1_000_000 // {N}), '%Y-%m-%d %H:%M:%S')
                   AS created_at,
               round(10 + (hash(i) % 30000) / 100, 2) AS amount,
               CASE WHEN i % 17 = 0 THEN 'cancelled' ELSE 'delivered' END AS status,
               'note from customer ' || (i % 1000) AS note
        FROM range({N}) t(i)""")
print(f"Output folder: {OUT}\napp database: {N:,} orders, {U.size_mb(APP):.0f} MB")

DuckDB is the course's **SQL engine**: it stores the tables, transforms them and serves the answers. Every other
tool on the map reads from it or writes to it.

In [ ]:
V.platform_map(tool="duckdb")

## 1 · The problem: analytics on the app database

Finance's question: **revenue per store per day**. The orders live in the app's database, a row store built to
save one order and fetch one customer quickly. The first try is to ask it directly.

In [ ]:
QUESTION = """
    SELECT substr(created_at, 1, 10) AS day, store_id, round(sum(amount), 2) AS revenue
    FROM orders
    WHERE status = 'delivered'
    GROUP BY 1, 2
    ORDER BY 1, 2"""

times = {}
app = sqlite3.connect(APP)
with U.timer("the app database (SQLite)", times):
    answer = app.execute(QUESTION).fetchall()
print(len(answer), "rows, first:", answer[0])

🔍 Seconds for one small chart, while customers place orders on the same database.

🤔 **Predict** · Second try: pull the table into pandas and sum there. Faster or slower? How much memory?

In [ ]:
with U.timer("pull into pandas", times):
    orders_df = pd.read_sql("SELECT * FROM orders", app)
with U.timer("compute in pandas", times):
    delivered = orders_df[orders_df.status == "delivered"]
    by_pandas = delivered.groupby([delivered.created_at.str[:10], "store_id"]).amount.sum()
print(f"pandas holds {orders_df.memory_usage(deep=True).sum() / 1e6:.0f} MB: every row and every column, "
      "including the notes we never use")
app.close()

🔍 The computing was quick; **moving** the data was slow. Every value left SQLite as a Python object, just to sum
one column.

Third try: copy the table once into a DuckDB file, then ask there.

In [ ]:
with duckdb.connect(f"{F}/analytics.duckdb") as con:
    with U.timer("copy into DuckDB (once)", times):
        con.sql(f"ATTACH '{APP.as_posix()}' AS app (TYPE sqlite, READ_ONLY)")
        con.sql("CREATE TABLE orders AS FROM app.orders")
    for attempt in range(3):
        with U.timer("the question in DuckDB", times if attempt == 2 else None):   # the third run is kept
            by_duckdb = con.sql(QUESTION).fetchall()

U.check(by_duckdb == answer, "the same answer as SQLite")
print(f"\nfile sizes: SQLite {U.size_mb(APP):.0f} MB · DuckDB {U.size_mb(OUT / 'analytics.duckdb'):.0f} MB")

In [ ]:
V.bars(pd.DataFrame({"way": list(times), "seconds": list(times.values())}), "way", ["seconds"],
       suptitle="Revenue per store per day: where the time goes", fmt="{:.2f}")

🔍 The same answer in a fraction of the time, from a file several times smaller. The data didn't change; only how
it is stored and processed did.

## 2 · The mental model: in-process, columnar, vectorized

```
 ROW STORE (SQLite, PostgreSQL)              COLUMN STORE (DuckDB, Parquet, Snowflake)
 page 1: [1, RUH-04, 08-01, 275.86, …, note] order_id: [1, 2, 3, 4, …]        ← skipped
 page 2: [2, JED-03, 08-01,  33.65, …, note] store_id: [RUH-04, JED-03, …]    ← read
 page 3: [3, RUH-04, 08-01,  12.00, …, note] amount:   [275.86, 33.65, …]     ← read
         sum(amount) reads every page        note:     [...]                  ← skipped
                                             similar values side by side compress well
```

| Idea | What it means | You saw it in §1 |
|---|---|---|
| **In-process** | a library inside your Python process, like SQLite; no server, no network | `duckdb.connect()`, nothing to install but the package |
| **Columnar** | each column stored apart, compressed | reads 3 of 6 columns; a smaller file |
| **Vectorized** | each operator works on batches of ~2,048 values, on all CPU cores | a GROUP BY over 1M rows in a blink |

| | Runs | Stores rows as | Built for |
|---|---|---|---|
| SQLite | in your process | rows | an app's own data: many small reads and writes |
| PostgreSQL, MySQL | a server | rows | many users writing small transactions |
| **DuckDB** | **in your process** | **columns** | **analytics on one machine** |
| Snowflake, BigQuery, Redshift | a managed service | columns | analytics on a cluster, shared by a company |

## 3 · A query, a connection, a database file

In [ ]:
duckdb.sql("SELECT 42 AS answer, 'مرحبا' AS greeting")

Three daily order exports from Qafila's stores. On the third day the export gained a column.

In [ ]:
FILES = {
    "orders_2026-09-14.csv": """order_id,store_id,status,amount,delivery_fee
101,RUH-04,delivered,275.86,9.00
102,JED-03,delivered,33.65,9.00
103,RUH-04,cancelled,12.00,0.00
""",
    "orders_2026-09-15.csv": """order_id,store_id,status,amount,delivery_fee
104,RUH-04,delivered,58.10,9.00
105,DMM-01,delivered,19.90,free
106,JED-03,delivered,44.00,9.00
""",
    "orders_2026-09-16.csv": """order_id,store_id,status,amount,delivery_fee,promo_code
107,RUH-04,delivered,8.75,9.00,YAWM96
108,JED-03,delivered,120.00,0.00,
109,DMM-01,delivered,61.25,9.00,YAWM96
110,RUH-04,cancelled,15.50,0.00,
""",
}
for name, text in FILES.items():
    (OUT / name).write_text(text, encoding="utf-8")
U.peek(OUT / "orders_2026-09-14.csv", 4)

In [ ]:
duckdb.sql(f"SELECT * FROM '{F}/orders_2026-09-14.csv'")

🔍 No `CREATE TABLE`, no import: a file path in `FROM` is a table.

`duckdb.sql()` uses a default connection that lives in memory. A pipeline opens its own **connection**, in memory
or to a **database file**.

🤔 **Predict** · We create a table in an in-memory connection, close it and connect again. Is the table there?

In [ ]:
mem = duckdb.connect()                 # in memory
mem.sql("CREATE TABLE hello AS SELECT 42 AS answer")
mem.close()

mem = duckdb.connect()
try:
    mem.sql("FROM hello")
except duckdb.CatalogException as e:
    print("❌", str(e).splitlines()[0])

In [ ]:
DB = OUT / "qafila_toy.duckdb"
with duckdb.connect(str(DB)) as con:   # a file: created if it doesn't exist
    con.sql("CREATE TABLE hello AS SELECT 42 AS answer")

with duckdb.connect(str(DB)) as con:
    print(con.sql("FROM hello").fetchall(), "| file:", DB.name, f"{U.size_mb(DB):.2f} MB")

🔍 An in-memory database disappears with its connection: right for exploring and for tests. A `.duckdb` file
keeps every table in one file, like SQLite. The course's warehouse is `warehouse/qafila.duckdb`.

| Code | Gives |
|---|---|
| `duckdb.sql(...)` | the default in-memory connection: quick looks |
| `duckdb.connect()` | a new in-memory database |
| `duckdb.connect("x.duckdb")` | a database file, created if missing |
| `duckdb.connect("x.duckdb", read_only=True)` | the same file, without the right to change it (§5) |
| `with duckdb.connect(...) as con:` | closes the connection at the end of the block |

## 4 · Files where they are: CSV, globs, JSON, Parquet

A CSV has no types, so DuckDB **sniffs** them from a sample of rows. `DESCRIBE` shows what it decided.

🤔 **Predict** · On 15 September one fee says `free`. Which type does `delivery_fee` get in that file?

In [ ]:
con = duckdb.connect()                 # in memory, for the rest of the notebook
for day in ["2026-09-14", "2026-09-15"]:
    types = con.sql(f"DESCRIBE FROM '{F}/orders_{day}.csv'").df()
    print(day, dict(zip(types.column_name, types.column_type)))

🔍 One value that isn't a number turns the whole column into `VARCHAR`, without an error. **Always check the types**
before trusting a file. Then take the decision yourself: state the type, and convert the odd values on purpose.

In [ ]:
con.sql(f"""
    SELECT order_id, delivery_fee AS as_sent,
           coalesce(TRY_CAST(delivery_fee AS DECIMAL(6, 2)), 0) AS delivery_fee_sar
    FROM read_csv('{F}/orders_2026-09-15.csv', types = {{'delivery_fee': 'VARCHAR'}})""")

🔍 `TRY_CAST` gives `NULL` where a cast fails, so the decision ("free" means 0) is written in the SQL, where it can
be read and tested.

A **glob** reads every daily file at once.

🤔 **Predict** · The third file has a `promo_code` column the others don't. What happens?

In [ ]:
con.sql(f"SELECT * FROM read_csv('{F}/orders_*.csv')")

🔍 No error, ten rows, and no `promo_code`. The columns came from the **first** file, and the third file's extra
column was silently dropped. (One `free` in one file also made `delivery_fee` text in all of them.)

`union_by_name = true` takes the columns of **all** files, matched by name, with `NULL` where a file lacks one.
`filename = true` adds the file each row came from, so you can trace a bad row back to its file.

In [ ]:
con.sql(f"""
    SELECT order_id, store_id, amount, promo_code, parse_filename(filename) AS file
    FROM read_csv('{F}/orders_*.csv', union_by_name = true, filename = true)
    ORDER BY order_id""")

➕ **JSON** keeps its nesting: an object becomes a `STRUCT` (read it with a dot), a list becomes a `LIST` (`unnest`
gives one row per element). The app's API sends orders like this:

In [ ]:
(OUT / "order_details.jsonl").write_text(
    '{"order_id": 104, "courier": {"id": "C-17", "vehicle": "bike"}, '
    '"items": [{"sku": "TEA-01", "qty": 2}, {"sku": "DATES-1KG", "qty": 1}]}\n'
    '{"order_id": 105, "courier": {"id": "C-22", "vehicle": "car"}, "items": [{"sku": "MILK-2L", "qty": 3}]}\n',
    encoding="utf-8")
display(con.sql(f"DESCRIBE FROM '{F}/order_details.jsonl'").df()[["column_name", "column_type"]])
con.sql(f"""
    SELECT order_id, courier.vehicle, unnest(items, recursive := true)
    FROM '{F}/order_details.jsonl'""")

➕ **Parquet** is the course's columnar file format (Lab 1.2). `COPY ... TO` writes any query to a file. Reading it
back needs no options, because the types are stored in the file.

In [ ]:
con.sql(f"""COPY (SELECT * FROM read_csv('{F}/orders_*.csv', union_by_name = true))
            TO '{F}/orders.parquet' (FORMAT parquet)""")
con.sql(f"DESCRIBE FROM '{F}/orders.parquet'").df()[["column_name", "column_type"]]

| Source | Read with | Notes |
|---|---|---|
| CSV | `FROM 'x.csv'`, `read_csv(...)` | types sniffed: check with `DESCRIBE` |
| JSON, JSON lines | `FROM 'x.json'`, `read_json(...)` | objects → `STRUCT`, lists → `LIST` |
| Parquet | `FROM 'x.parquet'`, `read_parquet(...)` | types stored in the file |
| Many files | `'orders_*.csv'`, `['a.csv', 'b.csv']` | `union_by_name`, `filename` |
| HTTP, S3, GCS, Azure | `FROM 'https://…'`, `'s3://…'` | the `httpfs` extension, loaded on first use |
| Write | `COPY (query) TO 'x.parquet'` | also CSV, JSON; `PARTITION_BY` in Lab 1.2 |

## 5 · Tables, views, other databases, and one writer at a time

A **table** is a copy, taken once. A **view** is a saved query, run again each time it is used.

In [ ]:
wh = duckdb.connect(str(DB))
wh.sql(f"CREATE TABLE orders AS FROM read_csv('{F}/orders_*.csv', union_by_name = true)")
wh.sql(f"CREATE VIEW orders_live AS FROM read_csv('{F}/orders_*.csv', union_by_name = true)")
wh.sql("SELECT (SELECT count(*) FROM orders) AS in_table, (SELECT count(*) FROM orders_live) AS in_view")

🤔 **Predict** · Thursday's export arrives. How many orders do the table and the view count now?

In [ ]:
(OUT / "orders_2026-09-17.csv").write_text(
    "order_id,store_id,status,amount,delivery_fee,promo_code\n111,JED-03,delivered,27.40,9.00,\n", encoding="utf-8")
wh.sql("SELECT (SELECT count(*) FROM orders) AS in_table, (SELECT count(*) FROM orders_live) AS in_view")

🔍 The table is fast and doesn't change under you; the view always shows the files as they are now. dbt (Day 2)
makes the same choice per model.

➕ **Other databases** can be attached and joined with yours, with no export and no import. That is how §1 copied
the app database: `ATTACH` with `TYPE sqlite` (PostgreSQL and MySQL work the same way).

In [ ]:
wh.sql(f"ATTACH '{APP.as_posix()}' AS app (TYPE sqlite, READ_ONLY)")
display(wh.sql("SELECT database_name, schema_name, table_name FROM duckdb_tables() ORDER BY ALL").df())
wh.sql("""
    SELECT o.store_id, count(*) AS toy_orders, any_value(a.n) AS orders_in_app
    FROM orders o
    JOIN (SELECT store_id, count(*) AS n FROM app.orders GROUP BY ALL) a USING (store_id)
    GROUP BY ALL ORDER BY ALL""")

**Break it on purpose.** Only **one process** at a time can change a `.duckdb` file. Our notebook holds `wh`, a
read-write connection to `qafila_toy.duckdb`. Now a second program (a dbt run, a dashboard, another notebook)
opens the same file.

🤔 **Predict** · It only wants to read. Does it get in?

In [ ]:
READER = f"import duckdb; print(duckdb.connect(r'{DB}', read_only=True).sql('SELECT count(*) FROM orders').fetchall()[0])"
other = subprocess.run([sys.executable, "-c", READER], capture_output=True, text=True)
print("❌" if other.returncode else "✅", other.stdout.strip() or first_error(other.stderr))

🔍 Refused: while one process may write, no other process may open the file, not even to read. Close our
connection and try again:

In [ ]:
wh.close()
other = subprocess.run([sys.executable, "-c", READER], capture_output=True, text=True)
print("❌" if other.returncode else "✅", other.stdout.strip() or first_error(other.stderr))

🔍 This is the most common DuckDB error in class: a notebook keeps the warehouse open, and `dbt run` or Dagster
fails with "Could not set lock on file" ("being used by another process" on Windows). To avoid it:
- open, query, close: `with duckdb.connect(path) as con:`;
- readers use `read_only=True` (many read-only processes can share a file);
- when many people or services must write, use a server (PostgreSQL, MotherDuck, a warehouse) or a table format
  like Iceberg (Day 3).

## 6 · Python in and out: DataFrames, relations, parameters

**In:** a pandas (or Polars, or Arrow) DataFrame in a Python variable can be queried by its **name**.

In [ ]:
targets = pd.DataFrame({"store_id": ["RUH-04", "JED-03", "DMM-01"], "daily_target": [150.0, 50.0, 35.0]})
con.sql("SELECT * FROM targets WHERE daily_target > 40")

**Out:** a result converts to whatever the next step needs.

In [ ]:
result = con.sql("SELECT store_id, daily_target FROM targets ORDER BY store_id")
print(type(result.df()).__module__, type(result.pl()).__module__, type(result.to_arrow_table()).__module__)
result.fetchall()

| Call | Gives |
|---|---|
| `.df()` | a pandas DataFrame |
| `.pl()` | a Polars DataFrame (Day 2) |
| `.to_arrow_table()` | an Arrow table (PyIceberg, Day 3) |
| `.fetchall()` | a list of Python tuples |
| `.show()` | prints a preview |

Older tutorials use `.fetchone()`, which relations lost in DuckDB 2.0: take `.fetchall()[0]` instead.

What `con.sql(...)` returns is a **relation**: a query that runs when you ask for rows, and again **every time**
you ask.

🤔 **Predict** · We keep a relation that counts the orders in the files, then Friday's file arrives. What does the
same relation say?

In [ ]:
order_count = con.sql(f"SELECT count(*) AS orders FROM read_csv('{F}/orders_*.csv', union_by_name = true)")
print("before:", order_count.fetchall())
(OUT / "orders_2026-09-18.csv").write_text(
    "order_id,store_id,status,amount,delivery_fee,promo_code\n112,DMM-01,delivered,33.00,9.00,\n", encoding="utf-8")
print("after: ", order_count.fetchall())

🔍 A relation behaves like a view: cheap to keep, re-run (glob included) when used. Call `.df()` once for a fixed
result. Relations also chain, the style Polars uses (Day 2):

In [ ]:
files = con.read_csv(f"{F}/orders_*.csv", union_by_name=True)
files.filter("status = 'delivered'").aggregate("store_id, round(sum(amount), 2) AS revenue").order("revenue DESC")

➕ **Parameters.** A value from Python (a date, a store picked in a dashboard) goes in as a parameter, not pasted into
the SQL string: no quoting bugs and no SQL injection.

In [ ]:
store = "JED-03"
con.execute(f"SELECT order_id, amount FROM read_csv('{F}/orders_*.csv', union_by_name = true) WHERE store_id = ?",
            [store]).df()

## 7 · Friendly SQL ➕

DuckDB's SQL is PostgreSQL-like, plus shortcuts the labs use.

In [ ]:
con.sql(f"CREATE OR REPLACE TABLE o AS FROM read_csv('{F}/orders_*.csv', union_by_name = true)")
con.sql("""
    FROM o                                         -- FROM first; SELECT * is implied
    SELECT * EXCLUDE (delivery_fee, promo_code)    -- every column but these
             REPLACE (round(amount) AS amount)     -- ...with this one changed
    WHERE status = 'delivered'
    LIMIT 3""")

In [ ]:
con.sql("""
    SELECT store_id, status, count(*) AS orders, max_by(order_id, amount) AS biggest_order
    FROM o
    GROUP BY ALL                                   -- group by every column that isn't aggregated
    ORDER BY ALL                                   -- order by every column, left to right""")

In [ ]:
con.sql("PIVOT o ON status USING round(sum(amount), 2) GROUP BY store_id ORDER BY store_id")

In [ ]:
con.sql("SUMMARIZE o").df()[["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"]]

| Shortcut | Does |
|---|---|
| `FROM t` / `FROM t SELECT ...` | `SELECT * FROM t` / FROM written first |
| `SELECT * EXCLUDE (a)`, `* REPLACE (expr AS a)` | all columns but some / with some changed |
| `GROUP BY ALL`, `ORDER BY ALL` | no need to repeat the columns |
| `COLUMNS('regex')` | the same expression on every matching column: `SELECT max(COLUMNS('amount|fee'))` |
| `max_by(x, y)`, `arg_min`, `FILTER (WHERE ...)` | the `x` of the largest `y`; conditional aggregates |
| `PIVOT`, `UNPIVOT` | rows ↔ columns |
| `DESCRIBE`, `SUMMARIZE` | the types / a profile of every column (Lab 1.1) |
| `QUALIFY` | filter on a window function (`00_sql_windows`, Lab 1.4) |

## 8 · Read the plan: EXPLAIN and EXPLAIN ANALYZE ➕

DuckDB turns your SQL into a **plan**, a tree of operators read **from the bottom up**. `EXPLAIN` shows it without
running the query. Here is §1's question, asked of a Parquet copy of the big table.

In [ ]:
with duckdb.connect(f"{F}/analytics.duckdb", read_only=True) as src:
    src.sql(f"COPY orders TO '{F}/big_orders.parquet' (FORMAT parquet)")
BIG = f"'{Path(os.path.relpath(OUT / 'big_orders.parquet')).as_posix()}'"   # relative: a shorter plan
print(con.sql(f"""EXPLAIN SELECT store_id, sum(amount) FROM {BIG}
                  WHERE status = 'delivered' GROUP BY ALL""").fetchall()[0][1])

🔍 The bottom box, the scan of the Parquet file, already knows two things from the query:
- **Projections**: only `store_id` and `amount` are read; the other columns are never touched.
- **Filters**: `status='delivered'` is applied while reading (a **pushed-down** filter).

Above it, a **Hash Group By** sums per store. Rows marked `~` are estimates, made before anything is read.

`EXPLAIN ANALYZE` **runs** the query and shows each operator's real rows and time.

In [ ]:
profile = con.sql(f"""EXPLAIN ANALYZE SELECT store_id, sum(amount) FROM {BIG}
                      WHERE status = 'delivered' GROUP BY ALL""").fetchall()[0][1]
print(re.search(r"Total Time: \S+", profile).group())
print(profile[profile.rindex("╭─"):])                     # the last box: the scan

🔍 The scan produced only the delivered orders: the cancelled ones never left it. When a query is slow, this shows
**where**: a scan reading too much (Lab 1.2), a join exploding, a sort spilling to disk.

🤔 **Predict** · DuckDB uses all CPU cores by default. How much slower is the same question on **one** thread?

In [ ]:
question = f"SELECT store_id, sum(amount) FROM {BIG} WHERE status = 'delivered' GROUP BY ALL"
cores = con.sql("SELECT current_setting('threads')").fetchall()[0][0]
all_cores = U.best_of(lambda: con.sql(question).fetchall())
con.sql("SET threads = 1")
one_core = U.best_of(lambda: con.sql(question).fetchall())
con.sql(f"SET threads = {cores}")
print(f"{cores} threads: {all_cores * 1000:.0f} ms | 1 thread: {one_core * 1000:.0f} ms | {one_core / all_cores:.1f}x")

🔍 Vectorized batches spread over every core: the parallelism a laptop gives for free. Beyond one machine you need
a cluster (Spark, Day 3).

## 9 · Summary

| | |
|---|---|
| **It does** | SQL analytics on one machine, fast · files in place (CSV, JSON, Parquet, Iceberg) · attach SQLite / PostgreSQL · DataFrames in and out · data larger than memory, by spilling to disk |
| **It doesn't** | serve an app (many small writes from many users: that's PostgreSQL) · let several processes write one file · scale past one machine (Spark, Day 3) · ingest, schedule or test by itself (dlt, Dagster, dbt) |
| **In this course** | the dlt destination (Lab 1.3) · the engine under dbt (Lab 2.3) · the warehouse `warehouse/qafila.duckdb` · a reader of Iceberg tables (Lab 3.1) |
| **As a service** | MotherDuck; Snowflake, BigQuery, Redshift and Databricks SQL offer the same columnar SQL at company scale |

**Cheat sheet**

| Code | Does |
|---|---|
| `duckdb.connect()` / `duckdb.connect("x.duckdb", read_only=True)` | in memory / a database file |
| `con.sql("...")` | a relation: runs when you ask for rows |
| `.df()`, `.pl()`, `.fetchall()` | results as pandas, Polars, tuples |
| `FROM 'x.csv'`, `read_csv(..., types = {...})` | a file as a table; take the types into your own hands |
| `read_csv('*.csv', union_by_name = true, filename = true)` | many files, matched by column name, traced to their file |
| `CREATE TABLE t AS ...`, `CREATE VIEW v AS ...` | a copy / a saved query |
| `ATTACH 'app.sqlite' AS app (TYPE sqlite, READ_ONLY)` | another database, joinable |
| `COPY (query) TO 'x.parquet' (FORMAT parquet)` | write a file |
| `con.execute("... WHERE id = ?", [value])` | a parameter from Python |
| `DESCRIBE`, `SUMMARIZE`, `EXPLAIN`, `EXPLAIN ANALYZE` | types, profile, plan, plan with timings |

## Checkpoint

1. DuckDB answered faster than both SQLite and pandas, from a smaller file. Why?
2. A column you expect to be numeric comes out as `VARCHAR`. What do you do, and what do you not do?
3. Daily files keep arriving: when is a table right, and when a view?
4. `dbt run` fails with "could not set lock on file". What is holding it, and what are two ways to avoid it?

Next: [Lab 1.1 · Platform tour with DuckDB](../D1_1_platform_tour.ipynb). Docs:
[Python API](https://duckdb.org/docs/stable/clients/python/overview) ·
[CSV import](https://duckdb.org/docs/stable/data/csv/overview) ·
[friendly SQL](https://duckdb.org/docs/stable/sql/dialect/friendly_sql) ·
[concurrency](https://duckdb.org/docs/stable/connect/concurrency) ·
[EXPLAIN](https://duckdb.org/docs/stable/guides/meta/explain).

**What you built:** the tables and the view in `qafila_toy.duckdb`.

In [ ]:
with duckdb.connect(str(DB), read_only=True) as db:
    V.database_map(db.sql("SELECT table_schema, table_name, table_type FROM information_schema.tables").df())